In [1]:
import pandas as pd
from pathlib import Path
from openai import OpenAI
import os
from dotenv import load_dotenv
import json

CSV_PATH = '../../asset/data/amazon_products_with_main_category.csv'
TEXT_COLUMN = "title"
LABEL_COLUMN = "main_category"
ID_COLUMN = "asin"
RANDOM_SEED = 30
PROMPT_UNKNOWN_LABEL = '__UNKNOWN__'
LIMIT_ROWS = 100000

load_dotenv()
client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)


df = pd.read_csv(
    CSV_PATH,
    usecols=["asin", "title", "main_category"],
)

print(f"Rows in amazon_products CSV: {len(df):,}")

# Remove missing values before string conversion
df = df.dropna(
    subset=[TEXT_COLUMN, LABEL_COLUMN]
)

# Clean text and label columns
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip()

df = (
    df[
        (df[TEXT_COLUMN] != "")
        & (df[LABEL_COLUMN] != "")
    ]
    .reset_index(drop=True)
)

# Deterministically sample rows after cleaning
if LIMIT_ROWS is not None and len(df) > LIMIT_ROWS:
    df = (
        df.sample(
            n=LIMIT_ROWS,
            random_state=RANDOM_SEED,
            replace=False,
        )
        .reset_index(drop=True)
    )

print(f"Rows used after cleaning and sampling: {len(df):,}")

df.head()

Rows in amazon_products CSV: 1,426,337
Rows used after cleaning and sampling: 100,000


,asin,title,main_category
0,B0BN1PTHGX,"Premium Jewelry Stand , Solid Clear 2-Tier Acr...",Home & Kitchen
1,B0C3TSRMBC,HPA200 Replacement Filters for Honeywell HPA20...,Home & Kitchen
2,B0CB88NJDN,"Couch Side Table with Adjustable Heights, Bamb...",Home & Kitchen
3,B089FKMQJR,Ironing Board Cover and Pad Extra Thick Heavy ...,Home & Kitchen
4,B082YS2K52,for Raspberry Pi 4 Aluminum Case with Fan and ...,Electronics & Computers


In [2]:
labels = sorted(df[LABEL_COLUMN].unique().tolist())
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

df["label"] = df[LABEL_COLUMN].map(label2id)

label_map = {
    "label2id": label2id,
    "id2label": {str(idx): label for idx, label in id2label.items()},
}

label_map

{'label2id': {'Arts, Crafts & Party Supplies': 0,
  'Automotive': 1,
  'Baby Products': 2,
  'Beauty & Personal Care': 3,
  'Electronics & Computers': 4,
  'Fashion, Shoes & Luggage': 5,
  'Gift Cards': 6,
  'Health & Household': 7,
  'Home & Kitchen': 8,
  'Industrial & Scientific': 9,
  'Pet Supplies': 10,
  'Smart Home': 11,
  'Sports & Outdoors': 12,
  'Tools & Home Improvement': 13,
  'Toys & Games': 14,
  'Video Games': 15},
 'id2label': {'0': 'Arts, Crafts & Party Supplies',
  '1': 'Automotive',
  '2': 'Baby Products',
  '3': 'Beauty & Personal Care',
  '4': 'Electronics & Computers',
  '5': 'Fashion, Shoes & Luggage',
  '6': 'Gift Cards',
  '7': 'Health & Household',
  '8': 'Home & Kitchen',
  '9': 'Industrial & Scientific',
  '10': 'Pet Supplies',
  '11': 'Smart Home',
  '12': 'Sports & Outdoors',
  '13': 'Tools & Home Improvement',
  '14': 'Toys & Games',
  '15': 'Video Games'}}

In [3]:
label_options = "\n".join(f"- {label}" for label in labels)

def build_prompt_only_category_prompt(title: str) -> str:
    return f"""
You are classifying product titles given product description.
Choose exactly one main_category from the allowed categories.
Return only the category name and no extra text.

Allowed categories:
{label_options}

Product title: {title}

main_category:
"""

def predict_prompt_only_main_category(
    title: str,
    model: str = "gpt-5.6-luna",
) -> dict[str, str | None]:

    prompt = build_prompt_only_category_prompt(title)

    response = client.responses.create(
        model=model,
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": "product_category",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "main_category": {
                            "type": "string",
                            "enum": labels,
                        }
                    },
                    "required": ["main_category"],
                    "additionalProperties": False,
                },
            }
        },
    )

    result = json.loads(response.output_text)

    predicted_label = result["main_category"]

    return {
        "generated_text": predicted_label,
        "predicted_label": predicted_label,
    }
    
# Quick sanity check
predict_prompt_only_main_category("Wireless Bluetooth Noise Cancelling Headphones with Microphone")

{'generated_text': 'Electronics & Computers',
 'predicted_label': 'Electronics & Computers'}

In [4]:
from tqdm.auto import tqdm

prompt_eval_sample = (
    df[
        df["main_category"].isin(labels)
        & df["title"].notna()
    ]
    .sample(
        n=min(100, len(df)),
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)

results = []

for _, row in tqdm(
    prompt_eval_sample.iterrows(),
    total=len(prompt_eval_sample),
    desc="Prompt-only evaluation"
):
    title = row["title"]

    try:
        prediction = predict_prompt_only_main_category(title)

        predicted_label = (
            prediction["predicted_label"]
            if prediction["predicted_label"] in labels
            else PROMPT_UNKNOWN_LABEL
        )

        generated_text = prediction["generated_text"]

    except Exception as e:
        predicted_label = PROMPT_UNKNOWN_LABEL
        generated_text = None

    results.append({
        "asin": row["asin"],
        "title": title,
        "true_main_category": row["main_category"],
        "predicted_main_category": predicted_label,
        "generated_text": generated_text,
    })


prompt_eval_results = pd.DataFrame(results)

Prompt-only evaluation:   0%|          | 0/100 [00:00<?, ?it/s]

In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
)

label_to_id = {
    label: idx
    for idx, label in enumerate(labels)
}

prompt_y_true = prompt_eval_results[
    "true_main_category"
].map(label_to_id)

prompt_y_pred = prompt_eval_results[
    "predicted_main_category"
].map(label_to_id)

unknown_id = len(labels)

prompt_y_pred_for_metrics = (
    prompt_y_pred
    .fillna(unknown_id)
    .astype(int)
)

prompt_y_true = prompt_y_true.astype(int)

prompt_only_metrics = {
    "eval_rows": int(len(prompt_eval_results)),

    "parse_success_rate": float(
        (
            prompt_eval_results["predicted_main_category"]
            != PROMPT_UNKNOWN_LABEL
        ).mean()
    ),

    "accuracy": float(
        accuracy_score(
            prompt_y_true,
            prompt_y_pred_for_metrics
        )
    ),

    "macro_f1": float(
        f1_score(
            prompt_y_true,
            prompt_y_pred_for_metrics,
            labels=list(range(len(labels))),
            average="macro",
            zero_division=0,
        )
    ),
}

prompt_only_metrics

{'eval_rows': 100,
 'parse_success_rate': 1.0,
 'accuracy': 0.76,
 'macro_f1': 0.6119565391004345}